# Week 33, Day 5: Neuromodulation: changing the rules of a circuit

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w033/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Less release, different message

Using the depressing synapse of Day 2 (τ_rec = 800 ms), compute the responses to 20 spikes at 20 Hz for U = 0.5 (`control`) and U = 0.2 (`modulated`), and compare the first and last responses and the ratio last/first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def dep_train(f, U, tau=800.0, n=20):
    x, out = 1.0, []
    for _ in range(n):
        out.append(U * x)
        x = 1 - (1 - (x - U * x)) * np.exp(-1000.0 / f / tau)
    return np.array(out)

trains = {"control": ..., "modulated": ...}
for name, a in trains.items():
    print(f"{name:9s}: first {a[0]:.3f}, twentieth {a[-1]:.3f}, ratio {a[-1] / a[0]:.2f}")
plt.plot(trains["control"], "o-", label="U = 0.5"); plt.plot(trains["modulated"], "s-", label="U = 0.2 (modulated)")
plt.xlabel("spike number"); plt.ylabel("response"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_first():
    "the modulated synapse's first response is 0.4 times the control's"
    assert abs(trains["modulated"][0] / trains["control"][0] - 0.4) < 1e-9

def test_late():
    "after 20 spikes the modulated response is within 25% of the control, so its train is more than twice as flat"
    assert trains["modulated"][-1] > 0.75 * trains["control"][-1]
    assert trains["modulated"][-1] / trains["modulated"][0] > 2 * trains["control"][-1] / trains["control"][0]

test_first()
test_late()
print("All checks passed")

### Exercise: Acetylcholine raises the gain

`alif(I, ga)` simulates 2 s of an adapting LIF neuron whose adaptation increment is scaled by ga, and returns its rate. Compute rates for currents of 300 to 600 pA with ga = 1 (control) and ga = 0.3 (acetylcholine), and the slope of each f-I curve in Hz per 100 pA, `gain[name]`.

In [ ]:
def alif(I, ga, T=2000.0, dt=0.1, tau_w=150.0, C=200.0, gL=10.0, EL=-70.0, Vth=-50.0, Vr=-60.0, Ew=-80.0):
    V, g, n = EL, 0.0, 0
    for _ in range(int(T / dt)):
        V += dt * (-gL * (V - EL) - g * (V - Ew) + I) / C
        g -= dt * g / tau_w
        if V >= Vth:
            V = Vr; g += ga; n += 1
    return n / (T / 1000)

currents = [300, 400, 500, 600]
rates, gain = {}, {}
for name, ga in (("control", 1.0), ("ACh", 0.3)):
    rates[name] = [alif(I, ga) for I in currents]
    gain[name] = ...
    print(f"{name:7s}: rates {rates[name]} Hz, gain {gain[name]:.1f} Hz per 100 pA")

In [ ]:
import numpy as np
def test_rates():
    "with reduced adaptation the neuron fires faster at every current tested"
    assert all(rates["ACh"][i] > rates["control"][i] for i in range(len(currents)))

def test_gain():
    "the slope of the f-I curve rises by more than half"
    assert gain["ACh"] > 1.5 * gain["control"]

test_rates()
test_gain()
print("All checks passed")